In [ ]:
import os
import random
import time
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
def unpickle(file):
    with open(file, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')
    return data_dict

def load(dir_path):
    s = 32 * 32
    x_train = []
    y_train = []
    for i in range(1, 6):
        data_dict = unpickle(os.path.join(dir_path, f"data_batch_{i}"))
        x_train.append(np.array([np.array([image[:s], image[s:s * 2], image[s * 2:s * 3]], dtype=np.uint8).T.reshape(32, 32, 3) for image in data_dict[b"data"]]))
        y_train.append(np.array(data_dict[b"labels"]))

    x_train = np.concatenate(x_train, axis=0)
    y_train = np.concatenate(y_train, axis=0)
    
    data_dict = unpickle(os.path.join(dir_path, f"test_batch"))
    x_test = np.array([np.array([image[:s], image[s:s * 2], image[s * 2:s * 3]], dtype=np.uint8).T.reshape(32, 32, 3) for image in data_dict[b"data"]])
    y_test = np.array(data_dict[b"labels"])

    return (torch.from_numpy(tensor) for tensor in (x_train, y_train, x_test, y_test))

def show_images(images, title_texts):
    cols = 5
    rows = math.ceil(len(images)/cols)
    plt.figure(figsize=(30,20))
    index = 1    
    for x in zip(images, title_texts):        
        image = x[0]        
        title_text = x[1]
        plt.subplot(rows, cols, index)        
        plt.imshow(image)
        if (title_text != ''):
            plt.title(title_text, fontsize = 15);        
        index += 1

In [ ]:
dir_path = "cifar-10-batches-py"
label_names = [label.decode("utf-8") for label in unpickle(os.path.join(dir_path, "batches.meta"))[b"label_names"]]
x_train, y_train, x_test, y_test = load(dir_path)

images_2_show = []
titles_2_show = []
for i in range(0, 10):
    r = random.randint(1, 50000)
    images_2_show.append(x_train[r - 1])
    titles_2_show.append('training image [' + str(r) + '] = ' + str(label_names[y_train[r - 1]]))

for i in range(0, 5):
    r = random.randint(1, 10000)
    images_2_show.append(x_test[r - 1])
    titles_2_show.append('test image [' + str(r) + '] = ' + str(label_names[y_test[r - 1]]))

show_images(images_2_show, titles_2_show)

In [ ]:
x_train = x_train.float().permute(0, 3, 1, 2) / 255.0
y_train = y_train.long()

x_test = x_test.float().permute(0, 3, 1, 2) / 255.0
y_test = y_test.long()

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.dropout = nn.Dropout(0.6)
        self.fc2 = nn.Linear(256, 10)
        self.relu = nn.ReLU()

        self.loss = nn.CrossEntropyLoss()
        self.opt = optim.Adam(self.parameters(), lr=0.001)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = x.reshape(-1, 64 * 8 * 8)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x
    
    def train_batch(self, x_batch, y_batch):
        outputs = self(x_batch)
        loss = self.loss(outputs, y_batch)

        self.opt.zero_grad()
        loss.backward()
        self.opt.step()

In [ ]:
def accuracy(y_pred, y_true):
    pred_labels = torch.argmax(y_pred, dim=1)
    return (pred_labels == y_true).float().mean()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN()
model = model.to(device)
x_train, y_train, x_test, y_test = (tensor.to(device) for tensor in (x_train, y_train, x_test, y_test))
print(device)

In [ ]:
epochs = 20

In [ ]:
loses = []
accurs = []

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=64, shuffle=False, num_workers=0)

model.train()
for epoch in range(1, epochs + 1):
    start_time = time.time()

    model.train()
    for x_batch, y_batch in train_loader:
        model.train_batch(x_batch, y_batch)
    
    model.eval()
    loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in train_loader:
            y_pred = model(x_batch)
            loss += model.loss(y_pred, y_batch).item() * x_batch.size(0)
            correct += (torch.argmax(y_pred, 1) == y_batch).sum().item()
            total += y_batch.size(0)
    loss /= total
    accur = correct / total
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch}/{epochs} — Loss: {loss:.4f}, Accuracy: {accur * 100:.2f}%, Time: {epoch_time:.2f}s")
    loses.append(loss)
    accurs.append(accur)

model.eval()
loss = 0
correct = 0
total = 0
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        y_pred = model(x_batch)
        loss += model.loss(y_pred, y_batch).item() * x_batch.size(0)
        correct += (torch.argmax(y_pred, 1) == y_batch).sum().item()
        total += y_batch.size(0)
loss /= total
accur = correct / total
print(f"\nTest — Loss: {loss:.4f}, Accuracy: {accur * 100:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(loses)
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Train-Loss')
ax1.set_title('Graph Loss(Epochs)')
ax1.grid(True)

ax2.plot(accurs)
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Train-Accuracy')
ax2.set_title('Graph  Accuracy(Epochs)')
ax2.grid(True)

plt.tight_layout()
plt.show()